In [1]:
!pip install zstandard

In [2]:
!pip install chess


In [3]:
import zstandard
import chess
import chess.pgn

print("Everything works")

Everything works


In [4]:
import os
os.listdir()

['.git',
 'Lichess data cleaning and analysis.ipynb',
 'Lichess data processing.ipynb',
 'README.md']

In [5]:
import zstandard as zstd
import os

# Base paths (Notice we leave the month as a placeholder {})
input_template = r"C:\Users\salma\OneDrive\Desktop\New folder\lichess_db_standard_rated_2013-{:02d}.pgn.zst"
output_template = r"D:\Chess\lichess_db_standard_rated_2013-{:02d}.pgn"

dctx = zstd.ZstdDecompressor()

# Loop from month 1 to 6
for month in range(1, 7):
    # This formats the number to be 2 digits (e.g., 1 becomes '01')
    input_file = input_template.format(month)
    output_file = output_template.format(month)
    
    print(f"Starting extraction for month {month:02d}...")
    
    try:
        with open(input_file, "rb") as compressed:
            with open(output_file, "wb") as decompressed:
                dctx.copy_stream(compressed, decompressed)
        print(f"Finished: {output_file} ✅")
    except FileNotFoundError:
        print(f"Error: Could not find {input_file}. Skipping...")

print("--- All extractions complete ---")

Starting extraction for month 01...
Finished: D:\Chess\lichess_db_standard_rated_2013-01.pgn ✅
Starting extraction for month 02...
Finished: D:\Chess\lichess_db_standard_rated_2013-02.pgn ✅
Starting extraction for month 03...
Finished: D:\Chess\lichess_db_standard_rated_2013-03.pgn ✅
Starting extraction for month 04...
Finished: D:\Chess\lichess_db_standard_rated_2013-04.pgn ✅
Starting extraction for month 05...
Finished: D:\Chess\lichess_db_standard_rated_2013-05.pgn ✅
Starting extraction for month 06...
Finished: D:\Chess\lichess_db_standard_rated_2013-06.pgn ✅
--- All extractions complete ---


In [ ]:

# Loop through months 1 to 6
for month in range(1, 7):
    filename = f"lichess_db_standard_rated_2013-{month:02d}.pgn"
    file_path = os.path.join(output_folder, filename)
    
    print(f"\n--- Checking File: {filename} ---")
    
    if os.path.exists(file_path):
        with open(file_path, encoding="utf-8") as pgn_file:
            # Read just the very first game to verify integrity
            first_game = chess.pgn.read_game(pgn_file)
            
            if first_game:
                print(f"✅ Success! First Game Headers:")
                print(f"   Event: {first_game.headers.get('Event')}")
                print(f"   Players: {first_game.headers.get('White')} vs {first_game.headers.get('Black')}")
                print(f"   Result: {first_game.headers.get('Result')}")
            else:
                print(f"⚠️ Warning: File exists but appears to be empty.")
    else:
        print(f"❌ Error: File not found at {file_path}")

print("\n--- All checks complete ---")


--- Checking File: lichess_db_standard_rated_2013-01.pgn ---
✅ Success! First Game Headers:
   Event: Rated Classical game
   Players: BFG9k vs mamalak
   Result: 1-0

--- Checking File: lichess_db_standard_rated_2013-02.pgn ---
✅ Success! First Game Headers:
   Event: Rated Classical game
   Players: Smok vs McCoy
   Result: 1-0

--- Checking File: lichess_db_standard_rated_2013-03.pgn ---
✅ Success! First Game Headers:
   Event: Rated Bullet game
   Players: esmir vs apsu2323
   Result: 0-1

--- Checking File: lichess_db_standard_rated_2013-04.pgn ---
✅ Success! First Game Headers:
   Event: Rated Bullet game
   Players: apsu2323 vs inhume
   Result: 1-0

--- Checking File: lichess_db_standard_rated_2013-05.pgn ---
✅ Success! First Game Headers:
   Event: Rated Bullet game
   Players: JesusLovesYou vs Azat
   Result: 0-1

--- Checking File: lichess_db_standard_rated_2013-06.pgn ---
✅ Success! First Game Headers:
   Event: Rated Bullet game
   Players: Kazuma vs kikeillana
   Result:

In [2]:
output_folder = r"D:\Chess"


In [10]:
import chess.pgn
import pandas as pd
import os

# Define your paths (using your variables from before)
output_folder = "D:\\Chess"
final_moves_csv_path = os.path.join(output_folder, "master_chess_moves_2013.csv")

for month in range(1, 7):
    filename = f"lichess_db_standard_rated_2013-{month:02d}.pgn"
    file_path = os.path.join(output_folder, filename)
    
    if os.path.exists(file_path):
        print(f"🚀 Extracting moves from {filename}...")
        month_data = []
        
        with open(file_path, encoding="utf-8") as pgn:
            while True:
                game = chess.pgn.read_game(pgn)
                if game is None:
                    break
                
                # 1. Get basic headers (ID or Players to link back to your first CSV)
                # It's good practice to keep 'White', 'Black', and 'Date' as a unique key
                game_info = {
                    "White": game.headers.get("White"),
                    "Black": game.headers.get("Black"),
                    "Date": game.headers.get("Date"),
                    "Result": game.headers.get("Result"),
                }
                
                # 2. Extract the actual moves as a single string
                # game.mainline_moves() is the move sequence
                # str(game.mainline_moves()) converts it to "1. e4 e5 2. Nf3..."
                game_info["Moves"] = str(game.mainline_moves())
                
                month_data.append(game_info)
        
        # 3. Save to CSV incrementally
        df_moves = pd.DataFrame(month_data)
        file_exists = os.path.isfile(final_moves_csv_path)
        df_moves.to_csv(final_moves_csv_path, mode='a', index=False, header=not file_exists, encoding="utf-8")
        
        print(f"✅ Saved {len(month_data)} games with moves from {filename}")
        
        # Free memory
        del month_data
        del df_moves
    else:
        print(f"Skipping {filename} (not found).")

print(f"\n🎉 Success! Your moves dataset is ready at: {final_moves_csv_path}")

🚀 Extracting moves from lichess_db_standard_rated_2013-01.pgn...


illegal san: 'a5' in r2kQb1r/pbpp3p/1pn1p3/7B/3PP2q/P1N5/1PP2PPP/R3K2R b KQ - 2 13 while parsing <Game at 0x1ff04403530 ('?' vs. '?', '????.??.??' at '?')>


✅ Saved 121333 games with moves from lichess_db_standard_rated_2013-01.pgn
🚀 Extracting moves from lichess_db_standard_rated_2013-02.pgn...


illegal san: 'e4' in r1b1kbr1/pp6/2p1pp2/7p/7N/8/PQP1BPPP/R4RK1 b q - 0 17 while parsing <Game at 0x1ff041840e0 ('?' vs. '?', '????.??.??' at '?')>


✅ Saved 123962 games with moves from lichess_db_standard_rated_2013-02.pgn
🚀 Extracting moves from lichess_db_standard_rated_2013-03.pgn...


...

✅ Saved 157872 games with moves from lichess_db_standard_rated_2013-04.pgn
🚀 Extracting moves from lichess_db_standard_rated_2013-05.pgn...


illegal san: 'e4' in 2kr1b2/p1np3p/bpp1p1p1/8/4P3/1P1P1P2/P1PRQ1BP/q1K3NR w - - 2 16 while parsing <Game at 0x1ff07e8b1d0 ('?' vs. '?', '????.??.??' at '?')>


✅ Saved 179551 games with moves from lichess_db_standard_rated_2013-05.pgn
🚀 Extracting moves from lichess_db_standard_rated_2013-06.pgn...


illegal san: 'd4' in 8/1R6/p1p2k2/2P2ppp/PP1P1P2/5K2/5P1P/8 b - - 0 32 while parsing <Game at 0x1ff0a540170 ('?' vs. '?', '????.??.??' at '?')>


✅ Saved 224680 games with moves from lichess_db_standard_rated_2013-06.pgn

🎉 Success! Your moves dataset is ready at: D:\Chess\master_chess_moves_2013.csv


In [11]:
import glob
import os

In [12]:
folder_path = r"D:\Chess"

# Use a broader pattern to find any CSV file with "moves" in the name
# This will catch 'master_chess_moves_2013.csv' and any others you generated
file_pattern = os.path.join(folder_path, "*moves*.csv") 
all_files = glob.glob(file_pattern)

# Debugging: Check if files were actually found
if not all_files:
    print(f"❌ Error: No files found in {folder_path} matching pattern: {file_pattern}")
    print("Check your folder to see the exact names of the CSVs.")
else:
    print(f"📂 Found {len(all_files)} files: {all_files}")
    
    # Read all files into a list
    df_list = [pd.read_csv(f) for f in all_files]
    
    # Combine into one giant DataFrame
    master_df = pd.concat(df_list, ignore_index=True)
    
    # Save the final result
    final_output = os.path.join(folder_path, "full_moves_dataset_combined_2013.csv")
    master_df.to_csv(final_output, index=False)
    
    print(f"✅ Success! Combined {len(all_files)} files into: {final_output}")
    print(f"📊 Total Rows: {len(master_df)}")

📂 Found 1 files: ['D:\\Chess\\master_chess_moves_2013.csv']
✅ Success! Combined 1 files into: D:\Chess\full_moves_dataset_combined_2013.csv
📊 Total Rows: 966034
